# Core Concepts in LangGraph

## Tutorials Covered:
1. State Management
2. Nodes and Edges
3. Conditional Logic
4. Parallel Processing

## 1. State Management

### Learning Objectives:
- Understand how state works in LangGraph
- Learn to define custom state structures
- Implement state updates in nodes

State management is fundamental to LangGraph, allowing applications to maintain context across multiple interactions and steps in a workflow.

In [ ]:
# Example of defining and using state in LangGraph
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph

# Define a more complex state structure
class State(TypedDict):
    # The current value in our state
    value: int
    # A list of messages that represent the conversation history
    messages: Annotated[list, operator.add]
    # A counter for tracking iterations
    iteration_count: int
    # A flag to indicate if we should continue processing
    continue_processing: bool

# Node that updates the state
def update_state(state):
    print(f"Processing iteration {state['iteration_count'] + 1}")
    updated_value = state["value"] + 1
    updated_messages = state["messages"] + [f"Updated value to {updated_value}"]
    updated_iteration = state["iteration_count"] + 1
    should_continue = updated_iteration < 3  # Stop after 3 iterations
    
    return {
        "value": updated_value,
        "messages": updated_messages,
        "iteration_count": updated_iteration,
        "continue_processing": should_continue
    }

# Create and compile the graph
workflow = StateGraph(State)
workflow.add_node("update", update_state)
workflow.set_entry_point("update")
workflow.set_finish_point("update")

app = workflow.compile()

# Test the graph with initial state
initial_state = {
    "value": 0,
    "messages": ["Starting process"],
    "iteration_count": 0,
    "continue_processing": True
}

result = app.invoke(initial_state)
print("Final state:", result)

## 2. Nodes and Edges

### Learning Objectives:
- Create and connect nodes in a graph
- Understand different types of nodes
- Implement various edge patterns

Nodes represent individual units of computation, while edges define how data flows between nodes in your LangGraph application.

In [ ]:
# Example of creating nodes and connecting them with edges
from langgraph.graph import StateGraph

# Define additional node functions
def double_value(state):
    new_value = state["value"] * 2
    return {"value": new_value, "messages": [f"Doubled value to {new_value}"]}

def square_value(state):
    new_value = state["value"] ** 2
    return {"value": new_value, "messages": [f"Squared value to {new_value}"]}

# Create a graph with multiple nodes
workflow = StateGraph(State)

# Add nodes to the graph
workflow.add_node("update", update_state)
workflow.add_node("double", double_value)
workflow.add_node("square", square_value)

# Set entry and finish points
workflow.set_entry_point("update")
workflow.set_finish_point("square")

# Add edges between nodes
workflow.add_edge("update", "double")
workflow.add_edge("double", "square")

app = workflow.compile()

# Test the multi-node graph
result = app.invoke(initial_state)
print("Result from multi-node graph:", result)

## 3. Conditional Logic

### Learning Objectives:
- Implement conditional branches in graphs
- Use conditional edges to control flow
- Create decision-making nodes

Conditional logic allows your LangGraph applications to make decisions and follow different execution paths based on state values.

In [ ]:
# Example of conditional logic in LangGraph
def route_by_value(state):
    """Route to different nodes based on the value in state"""
    if state["value"] % 2 == 0:
        return "even_handler"
    else:
        return "odd_handler"

def handle_even(state):
    new_value = state["value"] // 2  # Integer division by 2
    return {"value": new_value, "messages": [f"Even number divided by 2: {new_value}"]}

def handle_odd(state):
    new_value = state["value"] * 3 + 1  # Multiply by 3 and add 1
    return {"value": new_value, "messages": [f"Odd number processed: {new_value}"]}

# Create a graph with conditional routing
conditional_workflow = StateGraph(State)

conditional_workflow.add_node("update", update_state)
conditional_workflow.add_node("even_handler", handle_even)
conditional_workflow.add_node("odd_handler", handle_odd)

conditional_workflow.set_entry_point("update")
conditional_workflow.add_conditional_edges(
    "update",
    route_by_value,
    {
        "even_handler": "even_handler",
        "odd_handler": "odd_handler"
    }
)
conditional_workflow.add_edge("even_handler", "__end__")
conditional_workflow.add_edge("odd_handler", "__end__")

conditional_app = conditional_workflow.compile()

# Test with an even number
result_even = conditional_app.invoke({"value": 10, "messages": [], "iteration_count": 0, "continue_processing": True})
print("Result with even number:", result_even)

# Test with an odd number
result_odd = conditional_app.invoke({"value": 7, "messages": [], "iteration_count": 0, "continue_processing": True})
print("Result with odd number:", result_odd)

## 4. Parallel Processing

### Learning Objectives:
- Execute multiple nodes in parallel
- Handle concurrent operations in graphs
- Combine results from parallel nodes

Parallel processing allows multiple operations to be executed simultaneously, improving efficiency for independent tasks.

In [ ]:
# Example of parallel processing in LangGraph
import asyncio
import time

def slow_operation_1(state):
    # Simulate a slow operation
    time.sleep(1)
    result = state["value"] * 2
    return {"messages": [f"Slow operation 1 result: {result}"]}

def slow_operation_2(state):
    # Simulate another slow operation
    time.sleep(1)
    result = state["value"] ** 2
    return {"messages": [f"Slow operation 2 result: {result}"]}

def combine_results(state):
    # Combine results from parallel operations
    return {"value": state["value"] + 100, "messages": ["Results combined"]}

# Create a graph with parallel processing
parallel_workflow = StateGraph(State)

parallel_workflow.add_node("op1", slow_operation_1)
parallel_workflow.add_node("op2", slow_operation_2)
parallel_workflow.add_node("combine", combine_results)

parallel_workflow.set_entry_point("op1")
parallel_workflow.add_edge(["op1", "op2"], "combine")  # Fan-in pattern
parallel_workflow.add_edge("combine", "__end__")

parallel_app = parallel_workflow.compile()

# Test the parallel graph
start_time = time.time()
result_parallel = parallel_app.invoke({"value": 5, "messages": [], "iteration_count": 0, "continue_processing": True})
end_time = time.time()

print(f"Parallel processing result: {result_parallel}")
print(f"Execution time: {end_time - start_time:.2f} seconds")

## Practice Exercises

1. Create a state structure that tracks a shopping cart with items and total price
2. Build a graph with 4 nodes that implement a simple calculator (add, subtract, multiply, divide)
3. Implement a conditional graph that routes to different nodes based on user input type
4. Design a parallel processing workflow that performs different analysis on the same dataset

## Additional Resources

- [LangGraph State Management Guide](https://langchain-ai.github.io/langgraph/tutorials/more_complex_graphs/)
- [Conditional Edges in LangGraph](https://langchain-ai.github.io/langgraph/how-tos/conditional-edges/)
- [Parallel Processing Patterns](https://langchain-ai.github.io/langgraph/how-tos/parallel-processing/)